In [ ]:
# import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# load data
df_train = pd.read_csv("udel-churn-train.csv")
df_test = pd.read_csv("udel-churn-test.csv")
df_train.drop(['Unnamed: 0', 'customerID'], axis=1, inplace=True)

print(df_train.head())
print(df_train.info())

In [ ]:
# 数据清洗
df_train['TotalCharges'] = pd.to_numeric(df_train['TotalCharges'], errors='coerce')
df_test['TotalCharges'] = pd.to_numeric(df_test['TotalCharges'], errors='coerce')
df_train = df_train.dropna()
df_test = df_test.dropna()

In [ ]:
# EDA - Exploratory Data Analysis
print(df_train['Churn'].value_counts())
sns.countplot(x='Churn', data=df_train)
plt.title('Customer Churn Distribution')
plt.show()
df_train.hist(figsize=(10, 8))
plt.show()

In [ ]:
# 特征工程
X = df_train.drop('Churn', axis=1)
y = df_train['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

df_num = X_train.select_dtypes(exclude='object')
df_cat = X_train.select_dtypes(include='object')
num_features = df_num.columns.tolist()
cat_features = df_cat.columns.tolist()

num_pipeline = Pipeline(steps=[
    ('num_imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('cat_imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num_pipeline', num_pipeline, num_features),
    ('cat_pipeline', cat_pipeline, cat_features)
])

In [ ]:
# Baseline Model
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('baseline_clf', DummyClassifier(strategy='most_frequent'))
])

baseline_pipeline.fit(X_train, y_train)
baseline_pred = baseline_pipeline.predict(X_test)

print("Baseline Accuracy:", accuracy_score(y_test, baseline_pred))
print(classification_report(y_test, baseline_pred))

In [ ]:
# Logistic Regression
from sklearn.model_selection import cross_val_score

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('lr_clf', LogisticRegression(max_iter=1000))
])

lr_pipeline.fit(X_train, y_train)
lr_pred = lr_pipeline.predict(X_test)
print("Logistic Regression Accuracy:",accuracy_score(y_test, lr_pred))
print(classification_report(y_test, lr_pred))


# Cross Validation
cv_scores = cross_val_score(lr_pipeline, X, y, cv=5, scoring='accuracy')
print("Cross Validation Scores:")
print(cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

In [ ]:
# Decision Tree
dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('dt_clf', DecisionTreeClassifier(random_state=42))
])

param_grid = {
    'preprocessor__num_pipeline__num_imputer__strategy': ['mean', 'median'],
    'preprocessor__cat_pipeline__cat_imputer__strategy': ['most_frequent'],
    'dt_clf__criterion': ['gini', 'entropy'],
    'dt_clf__max_depth': [2,3,4,5,6]
}

grid_search = GridSearchCV(estimator=dt_pipeline, param_grid=param_grid, cv=10, scoring='accuracy')

grid_search.fit(X_train, y_train)
print(grid_search.best_params_)
print(grid_search.cv_results_['mean_test_score'].mean())

tree_clf_best = grid_search.best_estimator_

In [ ]:
# 评估模型性能
y_pred = tree_clf_best.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

In [ ]:
# ROC-AUC 指标
y_prob = tree_clf_best.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
print("Decision Tree ROC-AUC:", auc)

In [ ]:
# Random Forest
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('rf_clf', RandomForestClassifier(random_state=42))
])


rf_param_grid = {
    'rf_clf__n_estimators': [50, 100],
    'rf_clf__max_depth': [3, 5, 7],
    'rf_clf__criterion': ['gini', 'entropy']
}


rf_grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)


rf_grid_search.fit(X_train, y_train)
print("Best Random Forest Parameters:")
print(rf_grid_search.best_params_)

rf_best = rf_grid_search.best_estimator_

y_rf_pred = rf_best.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_rf_pred))

print(classification_report(y_test, y_rf_pred))

In [ ]:
# 模型比较
results = pd.DataFrame({
    'Model': [
        'Baseline',
        'Logistic Regression',
        'Decision Tree',
        'Random Forest'
    ],
    'Accuracy': [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_rf_pred)
    ]
})

print(results)

In [ ]:
# Feature Importance 特征重要性分析
encoded_cat_features = list(
    rf_best.named_steps['preprocessor']
    .transformers_[1][1]
    .named_steps['onehot']
    .get_feature_names_out(cat_features)
)

feature_names = num_features + encoded_cat_features

importances = rf_best.named_steps['rf_clf'].feature_importances_

feat_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

feat_importance = feat_importance.sort_values(
    by='Importance',
    ascending=False
)

print(feat_importance.head(10))

In [ ]:
# Feature Importance 可视化
plt.figure(figsize=(6, 3))
sns.barplot(x='Importance', y='Feature', data=feat_importance.head(10))
plt.title('Top 10 Important Features')
plt.show()

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Decision Tree Confusion Matrix')
plt.show()

In [ ]:
# 选择 Desicion Tree 模型进行最终预测
tree_clf_best = grid_search.best_estimator_
y_pred = tree_clf_best.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

X_test = pd.read_csv('udel-churn-test.csv')
X_test.drop(['Unnamed: 0', 'customerID'], axis=1, inplace=True)
X_test.TotalCharges = pd.to_numeric(X_test.TotalCharges, errors='coerce')

X_test = X_test[X_train.columns]

y_hat = tree_clf_best.predict(X_test)

submission = pd.DataFrame({
    'customerID': pd.read_csv('udel-churn-test.csv')['customerID'],
    'Churn': y_hat
})

submission.to_csv('submission_final.csv', index=False)